
 # 02 · Ingesta Bronze Review — YelpConnect

 **Tipo:** Pipeline de ingesta incremental  
 **Tecnología:** Databricks Auto Loader  
 **Entidad:** Review  
 **Patrón:** `availableNow=True`  
 **Destino:** `workspace.yelp_bronze.brz_review`

 ---

 ## Propósito

 Ingerir incrementalmente `review*.json.gz` conservando checkpoint
 y seguimiento de schema.

 Review se procesa de forma independiente porque es la entidad principal
 del caso Trust & Safety y la de mayor volumen.

 `availableNow=True` procesa los archivos pendientes y finaliza;
 no deja un stream ejecutándose permanentemente.

In [0]:
# COMMAND ----------

import os
import sys
import json
import uuid
import importlib.util

from pyspark.sql import functions as F


# ============================================================
# 1. CARGAR MÓDULO COMÚN BRONZE
# ============================================================

COMMON_MODULE_NAME = "bronze_common"

COMMON_MODULE_PATH = os.path.join(
    os.getcwd(),
    "bronze_common.py"
)


print("=" * 72)
print("YELPCONNECT - CARGA DE CONFIGURACIÓN BRONZE")
print("=" * 72)

print(
    f"Working directory : {os.getcwd()}"
)

print(
    f"Módulo común      : {COMMON_MODULE_PATH}"
)


if not os.path.isfile(
    COMMON_MODULE_PATH
):

    raise FileNotFoundError(
        f"""
        ❌ No se encontró el módulo común Bronze.

        Ruta esperada:
        {COMMON_MODULE_PATH}
        """
    )


spec = importlib.util.spec_from_file_location(
    COMMON_MODULE_NAME,
    COMMON_MODULE_PATH
)


if (
    spec is None
    or spec.loader is None
):

    raise RuntimeError(
        f"""
        ❌ No fue posible crear el loader para:

        {COMMON_MODULE_PATH}
        """
    )


bronze_common = (
    importlib.util.module_from_spec(
        spec
    )
)


sys.modules[
    COMMON_MODULE_NAME
] = bronze_common


spec.loader.exec_module(
    bronze_common
)


print(
    "✔ bronze_common.py cargado correctamente."
)


# ============================================================
# 2. EXPONER CONFIGURACIÓN COMPARTIDA
# ============================================================

CATALOG = bronze_common.CATALOG

BRONZE_SCHEMA = (
    bronze_common.BRONZE_SCHEMA
)

RAW = bronze_common.RAW

SOURCE_SYSTEM = (
    bronze_common.SOURCE_SYSTEM
)

LAYER = bronze_common.LAYER

METADATA_COLUMNS = (
    bronze_common.METADATA_COLUMNS
)


# ============================================================
# 3. EXPONER FUNCIONES COMPARTIDAS
# ============================================================

assert_bronze_schema_exists = (
    bronze_common
    .assert_bronze_schema_exists
)

add_bronze_metadata = (
    bronze_common
    .add_bronze_metadata
)


# ============================================================
# 4. CONFIGURACIÓN REVIEW
# ============================================================

PIPELINE_NAME = (
    "E7_01_bronze_review"
)

EXECUTION_ID = str(
    uuid.uuid4()
)


REVIEW_SOURCE = (
    f"{RAW}/review*.json.gz"
)

CHECKPOINT_BASE = (
    f"{RAW}/_checkpoints/review"
)

SCHEMA_LOCATION = (
    f"{CHECKPOINT_BASE}/schema"
)

CHECKPOINT_LOCATION = (
    f"{CHECKPOINT_BASE}/state"
)

REVIEW_TABLE = (
    f"{CATALOG}."
    f"{BRONZE_SCHEMA}."
    "brz_review"
)


# ============================================================
# 5. RESUMEN DE CONFIGURACIÓN
# ============================================================

print()
print("=" * 72)
print("YELPCONNECT - INGESTA BRONZE REVIEW")
print("=" * 72)

print(
    f"Pipeline         : {PIPELINE_NAME}"
)

print(
    f"Execution ID     : {EXECUTION_ID}"
)

print(
    f"Source           : {REVIEW_SOURCE}"
)

print(
    f"Schema Location  : {SCHEMA_LOCATION}"
)

print(
    f"Checkpoint       : {CHECKPOINT_LOCATION}"
)

print(
    f"Destino          : {REVIEW_TABLE}"
)

print("=" * 72)



 ## 1 · Precondición

 El schema Bronze debe haber sido creado por
 `00_setup/01_configurar_lakehouse`.
 El checkpoint no se elimina durante una ejecución normal.

In [0]:
assert_bronze_schema_exists(
    spark
)

print(
    f"✔ Schema disponible: "
    f"{CATALOG}.{BRONZE_SCHEMA}"
)








 ## 2 · Auto Loader

 Se utilizan `cloudFiles`, `schemaLocation`,
 `schemaEvolutionMode = addNewColumns`, checkpoint persistente y
 `availableNow=True`.

 Bronze no aplica lógica de negocio sobre Review.

In [0]:
# COMMAND ----------

df_review_raw = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(REVIEW_SOURCE)
)

df_review_bronze = add_bronze_metadata(
    df=df_review_raw,
    entity="review",
    execution_id=EXECUTION_ID,
    pipeline_name=PIPELINE_NAME,
    source_file_expression=F.col("_metadata.file_path")
)

query = (
    df_review_bronze
    .writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_LOCATION)
    .trigger(availableNow=True)
    .toTable(REVIEW_TABLE)
)

query.awaitTermination()

print("✔ Auto Loader finalizado.")

# COMMAND ----------

 ## 3 · Validación mínima

 La validación transversal de las cinco tablas se realiza posteriormente
 en `03_validar_bronze`

In [0]:
# COMMAND ----------

if not spark.catalog.tableExists(REVIEW_TABLE):
    raise RuntimeError(
        f"No existe la tabla {REVIEW_TABLE}."
    )

review_count = spark.table(REVIEW_TABLE).count()

if review_count == 0:
    raise RuntimeError(
        "brz_review quedó sin registros."
    )

print(
    f"✔ brz_review: {review_count:,} filas totales"
)

summary = {
    "status": "SUCCESS",
    "pipeline": PIPELINE_NAME,
    "execution_id": EXECUTION_ID,
    "table": REVIEW_TABLE,
    "rows_total": review_count,
    "checkpoint": CHECKPOINT_LOCATION
}

dbutils.notebook.exit(json.dumps(summary))
